# 04 小场景：实验室合作网络

用一个 6 人实验室、5 项成果的小数据集，串起全部公开 API：

| 结点 | 超边（成果） |
| --- | --- |
| `ann` `ben` `cy` `dee` `eli` `fay` | 3 篇论文 + 1 次三人项目 + 1 次双人实验 |

设定：超边 `weight` 表示该合作发生的次数（例如同一组合反复发表）。

**预期看到**：
1. `ann` 的**加权度最高**（5.0）——因为 `experiment-Y` 计了 3 次，而普通度只数到 3 条；
2. 投影后 `ann`–`ben` 与 `dee`–`fay` 看起来都是“一条边”，但前者来自两篇论文（paper-A、paper-C），后者只来自一个项目；
3. 用加权高阶度可以还原“合作强度”，而普通度只数条数。

In [1]:
import honet

# 硬编码数据：每条记录 = (成员, 标签, 权重/合作次数)
records = [
    (["ann", "ben", "cy"], "paper-A", 1),
    (["ben", "cy", "dee"], "paper-B", 1),
    (["ann", "ben", "cy", "dee"], "paper-C", 1),  # 四人合著
    (["dee", "eli", "fay"], "project-X", 1),
    (["ann", "fay"], "experiment-Y", 3),  # 同一组合做了 3 次
]

lab = honet.Hypergraph()
for members, label, weight in records:
    lab.add_edge(members, label=label, weight=weight)

print(lab)
for edge in lab.edges:
    print(f"  {edge.label:>12}: {sorted(edge.members)}  weight={edge.weight}")

<Hypergraph nodes=6 edges=5 uniform=False>
       paper-A: ['ann', 'ben', 'cy']  weight=1
       paper-B: ['ben', 'cy', 'dee']  weight=1
       paper-C: ['ann', 'ben', 'cy', 'dee']  weight=1
     project-X: ['dee', 'eli', 'fay']  weight=1
  experiment-Y: ['ann', 'fay']  weight=3


## 谁的合作面最广？

`hg.degree` 数“参与了几项成果”；这里再手算一个**加权度**（`Σ weight`），用来区分“参与次数”。
两者都基于同一批超边，只是读法不同。

In [2]:
print(f"{'成员':<5}{'高阶度':>6}{'加权度':>7}{'邻居数':>7}")
for node in lab.nodes:
    weighted = sum(e.weight for e in lab.edges if node in e)
    print(f"{node:<5}{lab.degree(node):>6}{weighted:>7.1f}{len(lab.neighbors(node)):>7}")

print()
print("度序列:", honet.degree_sequence(lab))
print("关联矩阵:", honet.incidence_matrix(lab))

成员      高阶度    加权度    邻居数
ann       3    5.0      4
ben       3    3.0      3
cy        3    3.0      3
dee       3    3.0      5
eli       1    1.0      2
fay       2    4.0      3

度序列: [3, 3, 3, 3, 2, 1]
关联矩阵: [[1, 0, 1, 0, 1], [1, 1, 1, 0, 0], [1, 1, 1, 0, 0], [0, 1, 1, 1, 0], [0, 0, 0, 1, 0], [0, 0, 0, 1, 1]]


## 投影：能告诉我们什么、丢掉了什么

投影后 `ann`–`cy` 相连（同属 paper-A），但**看不出**他们是“两人合作”还是“与另两人一起四人合著”。
下面把投影边逐条列出，并标注它们分别来自哪些超边，直观展示这种多对一的信息压缩。

In [3]:
proj = honet.clique_projection(lab)

# 每条投影边对应的来源超边（可能不止一条）
sources: dict[tuple[str, str], list[str]] = {}
for edge in lab.edges:
    for x in edge.members:
        for y in edge.members:
            if x < y:
                sources.setdefault((x, y), []).append(edge.label)

for pair in sorted(sources):
    print(f"{pair[0]}–{pair[1]:<4} 来源={sources[pair]}")

print()
print("投影边数:", len(sources))
print("ann 的投影邻居:", sorted(proj["ann"]), "但其中只有 fay 是纯二元合作")

ann–ben  来源=['paper-A', 'paper-C']
ann–cy   来源=['paper-A', 'paper-C']
ann–dee  来源=['paper-C']
ann–fay  来源=['experiment-Y']
ben–cy   来源=['paper-A', 'paper-B', 'paper-C']
ben–dee  来源=['paper-B', 'paper-C']
cy–dee  来源=['paper-B', 'paper-C']
dee–eli  来源=['project-X']
dee–fay  来源=['project-X']
eli–fay  来源=['project-X']

投影边数: 10
ann 的投影邻居: ['ben', 'cy', 'dee', 'fay'] 但其中只有 fay 是纯二元合作


## 小结

- **构造**：`Hypergraph(nodes=..., edges=...)` 或 `add_edge(members, label=..., weight=...)`，成员顺序被保留；
- **度量**：`incidence_matrix` 给出可核对的 0/1 矩阵，`degree_sequence` 给出降序高阶度；
- **降维**：`clique_projection` 把高阶关系变回成对网络，代价是丢失“同时出现”的分组信息；
- **加权**：`weight` 让“发生 3 次的二元合作”区别于“发生 1 次的三元合作”。

真实分析中，通常先用高阶度/关联矩阵判断是否值得保留高阶结构，再决定是否投影。